# Statistische Gesamtauswertung

Konsolidiertes Notebook: es vereint die (korrekte, log-basierte) statistische
Auswertung der beiden Abschluss-Funktionen (**Android**-Standard vs. **optimiert**)
mit den sekundaeren Masszahlen, dem Geschlechtervergleich und der
Personalisierungs-Analyse und erzeugt am Ende **eine** CSV
(`plots/eval_results.csv`), aus der die Thesis alle Zahlen liest.

Es werden **alle** JSON-Dateien im Ordner `ergebnisse/` verwendet;
Teilnehmerzahl und Geschlechter werden konkret gezaehlt.

Methodischer Kern:
1. **Log-Transformation** der Zeiten (`log(timeMs)`), Normalitaetspruefung.
2. **Gepaarter t-Test** auf den log-Zeiten (within-subject) inkl. Wilcoxon und $d_z$.
3. **Standardfehler & Konfidenzintervall** auf der log-Skala, rueck-transformiert
   in geometrische Mittel und einen multiplikativen Effekt, plus Bootstrap-CI.
4. **Zweifaktorielle Messwiederholungs-ANOVA** (Funktion x Distanz) mit
   **Mauchly-Test + Greenhouse-Geisser** (pingouin) und gemischtem Modell als Absicherung.
5. **Sekundaermasse**: Overshoot-Fehlerrate und Switchbacks (gepaart, McNemar).
6. **Vergleich nach Geschlecht** (nur bei ausreichender Gruppengroesse).
7. **Personalisierung**: Streuung der optimierten Parameter + gepoolter Median.
8. **CSV-Export** fuer die Thesis.

Ausfuehren mit dem globalen Python (`py`), das pandas/scipy/statsmodels/pingouin hat.


In [3]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import shapiro, skew, probplot, ttest_rel, wilcoxon, t as tdist


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'ergebnisse').is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
ERGEBNISSE_DIR = REPO_ROOT / 'ergebnisse'
PLOTS_DIR = REPO_ROOT / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

ANDROID_DEFAULTS = {
    'scrollFriction': 0.015,
    'decelerationRate': math.log(0.78) / math.log(0.9),
    'inflexion': 0.35,
}
PARAMS = ['scrollFriction', 'decelerationRate', 'inflexion']
PARAM_LABELS = {
    'scrollFriction': 'mu (scrollFriction)',
    'decelerationRate': 'r (decelerationRate)',
    'inflexion': 'beta (inflexion)',
}
PARAM_KEY = {'scrollFriction': 'Mu', 'decelerationRate': 'R', 'inflexion': 'Beta'}

BOUNDS_FILE = REPO_ROOT / 'src' / 'scrollPhysics' / 'parameterBounds.json'
with open(BOUNDS_FILE, encoding='utf-8') as fh:
    _b = json.load(fh)
PARAM_BOUNDS = {
    'scrollFriction': (float(_b['scrollFriction']['min']), float(_b['scrollFriction']['max'])),
    'decelerationRate': (float(_b['decelerationRate']['min']), float(_b['decelerationRate']['max'])),
    'inflexion': (float(_b['inflexion']['min']), float(_b['inflexion']['max'])),
}

ALPHA = 0.05
MIN_GROUP = 2            # Mindestgroesse je Geschlechtsgruppe fuer einen Vergleich
RNG = np.random.default_rng(20261005)


def save_fig(fig, name):
    out = PLOTS_DIR / name
    try:
        fig.savefig(out, dpi=150, bbox_inches='tight')
    except OSError as exc:
        print('WARN: konnte', name, 'nicht speichern:', exc)
    plt.close(fig)
    return out


def shapiro_safe(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return (float('nan'), float('nan'))
    if len(x) > 5000:
        x = RNG.choice(x, 5000, replace=False)
    res = shapiro(x)
    return (float(res.statistic), float(res.pvalue))


print('Repo:      ', REPO_ROOT)
print('Ergebnisse:', ERGEBNISSE_DIR)
print('Plots:     ', PLOTS_DIR)
print('Bounds:    ', PARAM_BOUNDS)


Repo:       C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app
Ergebnisse: C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\ergebnisse
Plots:      C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\plots
Bounds:     {'scrollFriction': (0.0025, 0.3), 'decelerationRate': (1.0, 15.0), 'inflexion': (0.05, 1.0)}


## 1. Daten laden und Abschluss-Funktionen bestimmen

Fuer jeden Teilnehmer bilden die beiden letzten Bloecke den Vergleich. Der
Android-Block wird robust an seinen Default-Parametern erkannt
($\mu=0.015$, $r\approx2.358$, $\beta=0.35$); der andere ist die optimierte
Funktion. Je Trial werden Zeit, Distanz, Overshoot und Switchbacks erfasst; aus
dem optimierten Block zusaetzlich der Parametervektor.


In [4]:
def is_android_params(ps, tol_f=1e-3, tol_r=1e-2, tol_i=1e-2):
    if not ps:
        return False
    try:
        return (
            abs(float(ps.get('scrollFriction')) - ANDROID_DEFAULTS['scrollFriction']) <= tol_f
            and abs(float(ps.get('decelerationRate')) - ANDROID_DEFAULTS['decelerationRate']) <= tol_r
            and abs(float(ps.get('inflexion')) - ANDROID_DEFAULTS['inflexion']) <= tol_i
        )
    except (TypeError, ValueError):
        return False


def display_name(p):
    pid = (p.get('prolificPid') or '').strip()
    return pid or (p.get('participantId') or 'unknown')


def final_condition_blocks(p):
    blocks = [b for b in (p.get('attemptBlocks') or []) if isinstance(b, dict)]
    blocks = sorted(blocks, key=lambda b: b.get('runNumber', 0))
    if len(blocks) < 2:
        return None
    last_two = blocks[-2:]
    android = opt = None
    for b in last_two:
        if is_android_params(b.get('parameterSet')):
            android = b
        else:
            opt = b
    if android is None or opt is None:
        group = p.get('group')
        first_block, second_block = last_two[0], last_two[1]
        if group == 1:
            opt, android = first_block, second_block
        else:
            opt, android = second_block, first_block
    return {'Android': android, 'Optimized': opt}


def _num_or_nan(v):
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return np.nan
    return float(v)


# Liest die JSON-Dateien DIREKT aus ergebnisse/ (nicht rekursiv; Unterordner werden ignoriert).
files = sorted(ERGEBNISSE_DIR.glob('*.json'))
rows = []
param_rows = []
skipped = []
for path in files:
    with open(path, encoding='utf-8') as fh:
        data = json.load(fh)
    for p in (data if isinstance(data, list) else [data]):
        conds = final_condition_blocks(p)
        if conds is None:
            skipped.append(display_name(p))
            continue
        name = display_name(p)
        gender = (p.get('gender') or 'unknown').strip().lower()
        ops = conds['Optimized'].get('parameterSet') or {}
        param_rows.append({'participant': name, 'gender': gender,
                           'scrollFriction': ops.get('scrollFriction'),
                           'decelerationRate': ops.get('decelerationRate'),
                           'inflexion': ops.get('inflexion')})
        for cond, block in conds.items():
            for a in (block.get('attempts') or []):
                tn = a.get('targetNumber')
                tm = a.get('timeMs')
                if tn in (None, 0) or tm is None or float(tm) <= 0:
                    continue
                ov = a.get('didOvershoot')
                rows.append({
                    'participant': name, 'gender': gender, 'condition': cond,
                    'distance': int(tn), 'timeMs': float(tm),
                    'didOvershoot': (1.0 if ov is True else 0.0 if ov is False else np.nan),
                    'switchbackCount': _num_or_nan(a.get('switchbackCount')),
                })

if not rows:
    raise SystemExit('Keine Trial-Daten gefunden. Bitte die JSON-Dateien DIREKT in '
                     + str(ERGEBNISSE_DIR) + ' ablegen (nicht in Unterordner).')

trials = pd.DataFrame(rows)
trials['logTime'] = np.log(trials['timeMs'])
opt_params = pd.DataFrame(param_rows)
for col in PARAMS:
    opt_params[col] = pd.to_numeric(opt_params[col], errors='coerce')

N_PARTICIPANTS = int(trials['participant'].nunique())
gender_counts = trials.drop_duplicates('participant')['gender'].value_counts().to_dict()

print('Dateien im Ordner ergebnisse/ :', len(files))
print('Ausgewertete Teilnehmer       :', N_PARTICIPANTS)
print('Geschlechterverteilung        :', gender_counts)
print('Distanzstufen                 :', sorted(trials['distance'].unique()))
print('Trials gesamt                 :', len(trials))
if skipped:
    print('Uebersprungen (weniger als 2 Bloecke):', skipped)
trials.head()


Dateien im Ordner ergebnisse/ : 10
Ausgewertete Teilnehmer       : 10
Geschlechterverteilung        : {'male': 6, 'female': 4}
Distanzstufen                 : [np.int64(30), np.int64(60), np.int64(90), np.int64(120), np.int64(150), np.int64(180), np.int64(210), np.int64(240), np.int64(270), np.int64(300)]
Trials gesamt                 : 200


,participant,gender,condition,distance,timeMs,didOvershoot,switchbackCount,logTime
0,6a968836c2a369c9285714da,female,Android,120,3192.0,1.0,1.0,8.068403
1,6a968836c2a369c9285714da,female,Android,210,7095.0,0.0,0.0,8.867146
2,6a968836c2a369c9285714da,female,Android,270,9947.0,0.0,0.0,9.205026
3,6a968836c2a369c9285714da,female,Android,90,3493.0,1.0,0.0,8.158516
4,6a968836c2a369c9285714da,female,Android,60,2440.0,1.0,0.0,7.799753


## 2. Log-Transformation und Verteilungspruefung

Rohe Zeiten sind rechtsschief; die Log-Transformation reduziert die Schiefe. Die
gepoolte Shapiro-Pruefung ist nur **deskriptiv**; die eigentliche Testannahme
wird in Abschnitt 3 auf den gepaarten Differenzen geprueft.


In [5]:
raw = trials['timeMs'].to_numpy(float)
logt = trials['logTime'].to_numpy(float)

SKEW_RAW = float(skew(raw))
SKEW_LOG = float(skew(logt))
W_RAW = shapiro_safe(raw)
W_LOG = shapiro_safe(logt)
print('Schiefe (Skewness):')
print('  timeMs      :', round(SKEW_RAW, 3))
print('  log(timeMs) :', round(SKEW_LOG, 3))
print()
print('Shapiro-Wilk (gepoolte Trials, deskriptiv):')
print('  timeMs      : W=%.3f  p=%.4g' % W_RAW)
print('  log(timeMs) : W=%.3f  p=%.4g' % W_LOG)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(raw, bins=30, color='#999999')
axes[0].set_title('timeMs (roh)')
axes[0].set_xlabel('Zeit (ms)')
axes[0].set_ylabel('Haeufigkeit')
axes[1].hist(logt, bins=30, color='#3b7dd8')
axes[1].set_title('log(timeMs)')
axes[1].set_xlabel('log Zeit')
fig.suptitle('Verteilung der Zeiten vor/nach Log-Transformation')
save_fig(fig, 'eval_time_distribution.png')
plt.show()


Schiefe (Skewness):
  timeMs      : 0.893
  log(timeMs) : -0.356

Shapiro-Wilk (gepoolte Trials, deskriptiv):
  timeMs      : W=0.944  p=5.093e-07
  log(timeMs) : W=0.979  p=0.004947


## 3. Normalitaet der gepaarten Differenzen

Der gepaarte t-Test setzt annaehernd normalverteilte **Differenzen** voraus. Je
Proband wird der Mittelwert der log-Zeiten pro Funktion gebildet und die Differenz
(Optimized $-$ Android) mit Shapiro-Wilk und Q-Q-Plot geprueft.


In [6]:
agg = (trials.groupby(['participant', 'condition'])['logTime'].mean()
       .unstack('condition'))
for needed in ('Android', 'Optimized'):
    if needed not in agg.columns:
        raise ValueError('Bedingung fehlt in den Daten: ' + needed)
agg = agg.dropna(subset=['Android', 'Optimized'])
d = (agg['Optimized'] - agg['Android']).to_numpy(float)

print('Gepaarte Teilnehmer (beide Funktionen vorhanden):', len(agg))
W_DIFF = shapiro_safe(d)
print('Shapiro-Wilk auf den gepaarten log-Differenzen: W=%.3f  p=%.4g' % W_DIFF)
if not math.isnan(W_DIFF[1]) and W_DIFF[1] >= ALPHA:
    print('-> p >= %.2f: Normalverteilung der Differenzen wird nicht verworfen; t-Test vertretbar.' % ALPHA)
else:
    print('-> p < %.2f oder n zu klein: Normalitaet fraglich; Wilcoxon-Test dient als Absicherung.' % ALPHA)

fig, ax = plt.subplots(figsize=(5, 5))
probplot(d, dist='norm', plot=ax)
ax.set_title('Q-Q-Plot der gepaarten Differenzen (log-Zeit)')
save_fig(fig, 'eval_qq_paired_diff.png')
plt.show()


Gepaarte Teilnehmer (beide Funktionen vorhanden): 10
Shapiro-Wilk auf den gepaarten log-Differenzen: W=0.957  p=0.7553
-> p >= 0.05: Normalverteilung der Differenzen wird nicht verworfen; t-Test vertretbar.


## 4. Gepaarter t-Test auf den log-Zeiten (Android vs. optimiert)

Within-subject, daher **gepaart**. Getestet wird auf den pro Proband gemittelten
log-Zeiten; Wilcoxon und Cohen's $d_z$ zusaetzlich.


In [7]:
tt = ttest_rel(agg['Optimized'], agg['Android'])
n = len(agg)
sd_d = float(np.std(d, ddof=1))
dz = float(np.mean(d) / sd_d) if sd_d > 0 else float('nan')
try:
    wil = wilcoxon(agg['Optimized'], agg['Android'])
    WILCOXON_P = float(wil.pvalue)
except Exception as exc:
    WILCOXON_P = float('nan')
    print('Wilcoxon nicht berechenbar:', exc)

print('Gepaarter t-Test (log-Zeit, Optimized vs. Android):')
print('  n       :', n)
print('  t       : %.4f' % tt.statistic)
print('  df      :', n - 1)
print('  p-Wert  : %.5f' % tt.pvalue)
print('  Cohen dz: %.3f' % dz)
print('  Wilcoxon p: %.5f' % WILCOXON_P)


Gepaarter t-Test (log-Zeit, Optimized vs. Android):
  n       : 10
  t       : -2.8388
  df      : 9
  p-Wert  : 0.01944
  Cohen dz: -0.898
  Wilcoxon p: 0.03711


## 5. Standardfehler, Konfidenzintervall und geometrische Mittel

**Ueberlegung.** Analyse auf $y=\log(\text{timeMs})$, Differenzen je Proband
$d_i=\bar y_{\text{opt},i}-\bar y_{\text{And},i}$.

- **Standardfehler**: $\mathrm{SE}=s_d/\sqrt{n}$.
- **95%-CI** (t, $n-1$ df): $\bar d\pm t_{n-1,0.975}\,\mathrm{SE}$.
- **Rueck-Transformation**: $\exp(\bar d)$ = Verhaeltnis der geometrischen Mittel
  (Opt/And); CI durch Exponentiation. Je Funktion ergibt sich das geometrische
  Mittel in ms und dessen CI analog.
- Zusaetzlich ein **Bootstrap-CI** (verteilungsfrei).


In [8]:
mean_d = float(np.mean(d))
se = sd_d / math.sqrt(n)
tcrit = float(tdist.ppf(0.975, n - 1))
ci_lo, ci_hi = mean_d - tcrit * se, mean_d + tcrit * se
ratio = math.exp(mean_d)
ratio_lo, ratio_hi = math.exp(ci_lo), math.exp(ci_hi)

GM = {}
GM_CI = {}
for cond in ('Android', 'Optimized'):
    vals = agg[cond].to_numpy(float)
    m = float(np.mean(vals))
    se_c = float(np.std(vals, ddof=1)) / math.sqrt(len(vals))
    tc = float(tdist.ppf(0.975, len(vals) - 1))
    GM[cond] = math.exp(m)
    GM_CI[cond] = (math.exp(m - tc * se_c), math.exp(m + tc * se_c))

boot = np.array([np.mean(RNG.choice(d, size=n, replace=True)) for _ in range(10000)])
BOOT_RATIO = (math.exp(np.percentile(boot, 2.5)), math.exp(np.percentile(boot, 97.5)))

print('Effekt auf der log-Skala (Optimized - Android):')
print('  mittlere Differenz : %.4f' % mean_d)
print('  Standardfehler SE  : %.4f' % se)
print('  95%% CI            : [%.4f, %.4f]' % (ci_lo, ci_hi))
print()
print('Multiplikativer Effekt: Verhaeltnis Opt/And = %.4f  (95%% CI [%.4f, %.4f]) -> %+.1f %%' % (
    ratio, ratio_lo, ratio_hi, (ratio - 1) * 100))
print('Bootstrap-95%%-CI (Verhaeltnis): [%.4f, %.4f]' % BOOT_RATIO)
print()
print('Geometrische Mittel pro Funktion:')
for cond in ('Android', 'Optimized'):
    print('  %-9s: GM = %.1f ms   95%% CI [%.1f, %.1f] ms' % (
        cond, GM[cond], GM_CI[cond][0], GM_CI[cond][1]))

fig, ax = plt.subplots(figsize=(5, 4))
conds = ['Android', 'Optimized']
means = [GM[c] for c in conds]
lower = [GM[c] - GM_CI[c][0] for c in conds]
upper = [GM_CI[c][1] - GM[c] for c in conds]
ax.bar(conds, means, yerr=[lower, upper], capsize=6, color=['#888888', '#3b7dd8'])
ax.set_ylabel('Geometrisches Mittel der Zeit (ms)')
ax.set_title('Android vs. Optimized (n = %d), 95%% CI' % n)
save_fig(fig, 'eval_android_vs_optimized_overall.png')
plt.show()


Effekt auf der log-Skala (Optimized - Android):
  mittlere Differenz : -0.1357
  Standardfehler SE  : 0.0478
  95% CI            : [-0.2438, -0.0276]

Multiplikativer Effekt: Verhaeltnis Opt/And = 0.8731  (95% CI [0.7837, 0.9728]) -> -12.7 %
Bootstrap-95%-CI (Verhaeltnis): [0.8028, 0.9583]

Geometrische Mittel pro Funktion:
  Android  : GM = 4242.8 ms   95% CI [3591.9, 5011.7] ms
  Optimized: GM = 3704.6 ms   95% CI [3271.9, 4194.5] ms


## 6. Zweifaktorielle Messwiederholungs-ANOVA (Funktion x Distanz)

Beide unabhaengigen Variablen sind **innerhalb** der Probanden variiert:
**Funktion** (2 Stufen) und **Distanz** (10 Zielpositionen). Eine Zeile pro
Proband und Zelle; abhaengige Variable `log(timeMs)`. Es gehen nur Probanden mit
**vollstaendigem** Design ($2\times10$ Zellen) ein.

Sphaerizitaet wird mit dem **Mauchly-Test** geprueft; bei Verletzung werden
**Greenhouse-Geisser-korrigierte** p-Werte (`pingouin`) berichtet. Ein lineares
gemischtes Modell dient als Absicherung ohne Sphaerizitaetsannahme.


In [9]:
from statsmodels.stats.anova import AnovaRM

rm = (trials.groupby(['participant', 'condition', 'distance'])['logTime']
      .mean().reset_index())
n_dist = rm['distance'].nunique()
cell_counts = rm.groupby('participant').size()
complete = cell_counts[cell_counts == 2 * n_dist].index
rm_c = rm[rm['participant'].isin(complete)].copy()

print('Distanzstufen:', n_dist)
print('Vollstaendige Teilnehmer (2 x %d Zellen):' % n_dist, len(complete), 'von', len(cell_counts))
incomplete = sorted(set(cell_counts.index) - set(complete))
if incomplete:
    print('  ausgeschlossen (unvollstaendiges Design):', incomplete)
print()

try:
    aov = AnovaRM(rm_c, depvar='logTime', subject='participant',
                  within=['condition', 'distance'], aggregate_func='mean').fit()
    print('Zweifaktorielle RM-ANOVA auf log(timeMs):')
    print(aov)
except Exception as exc:
    aov = None
    print('RM-ANOVA nicht berechenbar (evtl. zu wenige vollstaendige Teilnehmer):', exc)


Distanzstufen: 10
Vollstaendige Teilnehmer (2 x 10 Zellen): 10 von 10

Zweifaktorielle RM-ANOVA auf log(timeMs):
                     Anova
                   F Value Num DF  Den DF Pr > F
------------------------------------------------
condition           8.0590 1.0000  9.0000 0.0194
distance           67.4308 9.0000 81.0000 0.0000
condition:distance  1.7169 9.0000 81.0000 0.0984



In [10]:
import pingouin as pg

PG = {}
MAUCHLY = (float('nan'), float('nan'))
try:
    pg_aov = pg.rm_anova(data=rm_c, dv='logTime', within=['condition', 'distance'],
                         subject='participant', detailed=True, effsize='np2')
    cols = [c for c in ['Source', 'ddof1', 'ddof2', 'F', 'p_unc',
                        'p_GG_corr', 'np2', 'eps', 'W_spher', 'p_spher', 'sphericity']
            if c in pg_aov.columns]
    print('pingouin RM-ANOVA auf log(timeMs) mit Sphaerizitaet / Greenhouse-Geisser:')
    print(pg_aov[cols].round(5).to_string(index=False))
    print()
    print('Lesehilfe: eps = Greenhouse-Geisser-Epsilon, p_GG_corr = GG-korrigiert,')
    print('p_spher = Mauchly (p<0.05 -> Sphaerizitaet verletzt -> p_GG_corr verwenden).')
    label = {'condition': 'Function', 'distance': 'Distance', 'condition * distance': 'Interaction'}
    for _, r in pg_aov.iterrows():
        key = label.get(str(r['Source']))
        if key:
            PG[key] = {c: r.get(c) for c in ['F', 'ddof1', 'ddof2', 'p_unc', 'p_GG_corr', 'eps', 'np2']}
except Exception as exc:
    print('pingouin RM-ANOVA nicht berechenbar:', exc)

print()
try:
    spher = pg.sphericity(data=rm_c, dv='logTime', within=['distance'], subject='participant')
    MAUCHLY = (float(spher.W), float(spher.pval))
    verdict = ('verletzt -> Greenhouse-Geisser verwenden' if spher.pval < ALPHA
               else 'nicht verworfen')
    print('Mauchly-Test (Faktor Distanz): W=%.4f  p=%.4g  -> Sphaerizitaet %s' % (
        MAUCHLY[0], MAUCHLY[1], verdict))
except Exception as exc:
    print('Mauchly-Test (Distanz) nicht berechenbar:', exc)


pingouin RM-ANOVA auf log(timeMs) mit Sphaerizitaet / Greenhouse-Geisser:
              Source  ddof1  ddof2        F   p_unc  p_GG_corr     np2     eps  W_spher  p_spher  sphericity
           condition      1      9  8.05904 0.01944    0.01944 0.47242 1.00000      NaN  1.00000        True
            distance      9     81 67.43085 0.00000    0.00000 0.88225 0.26342  0.00000  0.01549       False
condition * distance      9     81  1.71695 0.09838    0.17846 0.16021 0.38135  0.00001  0.02547       False

Lesehilfe: eps = Greenhouse-Geisser-Epsilon, p_GG_corr = GG-korrigiert,
p_spher = Mauchly (p<0.05 -> Sphaerizitaet verletzt -> p_GG_corr verwenden).

Mauchly-Test (Faktor Distanz): W=0.0000  p=0.01531  -> Sphaerizitaet verletzt -> Greenhouse-Geisser verwenden


In [11]:
import statsmodels.formula.api as smf

RESID_SHAPIRO = (float('nan'), float('nan'))
LMM_FUNCTION = (float('nan'), float('nan'))  # (coef, p) additiv
try:
    lmm_full = smf.mixedlm('logTime ~ C(condition) * C(distance)', rm_c,
                           groups=rm_c['participant']).fit()
    resid = np.asarray(lmm_full.resid, float)
    RESID_SHAPIRO = shapiro_safe(resid)
    print('Residuen-Normalitaet (volles Modell, Shapiro): W=%.3f  p=%.4g' % RESID_SHAPIRO)
    fig, ax = plt.subplots(figsize=(5, 5))
    probplot(resid, dist='norm', plot=ax)
    ax.set_title('Q-Q-Plot der Modell-Residuen (log-Zeit)')
    save_fig(fig, 'eval_qq_residuals.png')
    plt.show()
except Exception as exc:
    print('Residuencheck nicht berechenbar:', exc)

try:
    lmm_main = smf.mixedlm('logTime ~ C(condition) + C(distance)', rm_c,
                           groups=rm_c['participant']).fit()
    coef = float(lmm_main.params.get('C(condition)[T.Optimized]', float('nan')))
    pval = float(lmm_main.pvalues.get('C(condition)[T.Optimized]', float('nan')))
    LMM_FUNCTION = (coef, pval)
    print()
    print('LMM-Kreuzcheck Haupteffekt Funktion (additiv, ohne Sphaerizitaetsannahme):')
    print('  Koeffizient (Optimized vs. Android): %.4f  p=%.4g  -> exp=%.4f (%+.1f %%)' % (
        coef, pval, math.exp(coef), (math.exp(coef) - 1) * 100))
except Exception as exc:
    print('Additives gemischtes Modell nicht berechenbar:', exc)


Residuen-Normalitaet (volles Modell, Shapiro): W=0.948  p=1.095e-06

LMM-Kreuzcheck Haupteffekt Funktion (additiv, ohne Sphaerizitaetsannahme):
  Koeffizient (Optimized vs. Android): -0.1357  p=8.852e-05  -> exp=0.8731 (-12.7 %)


## 7. Interaktions-Plot: Distanzprofile beider Funktionen

Mittlere log-Zeit je Distanz und Funktion (Fehlerbalken = Standardfehler ueber
Probanden).


In [12]:
prof = (rm_c.groupby(['condition', 'distance'])['logTime']
        .agg(['mean', 'sem']).reset_index())
fig, ax = plt.subplots(figsize=(8, 5))
for cond, color in (('Android', '#888888'), ('Optimized', '#3b7dd8')):
    sub = prof[prof['condition'] == cond].sort_values('distance')
    ax.errorbar(sub['distance'], sub['mean'], yerr=sub['sem'].fillna(0),
                marker='o', capsize=4, color=color, label=cond)
ax.set_xlabel('Zieldistanz (Index)')
ax.set_ylabel('Mittlere log(timeMs)')
ax.set_title('Funktion x Distanz (log-Zeit, Mittel +/- SE)')
ax.legend()
save_fig(fig, 'eval_android_vs_optimized_by_distance.png')
plt.show()


## 8. Sekundaermasse: Overshoot-Fehlerrate und Switchbacks

Fehlerrate = Anteil Trials mit Overshoot (`didOvershoot`); Switchbacks =
Richtungswechsel pro Trial. Pro Proband je Funktion gemittelt und gepaart getestet
(Wilcoxon). Zusaetzlich McNemar auf der gepaarten 2x2-Tabelle pro (Proband, Distanz).


In [13]:
sec = (trials.groupby(['participant', 'condition'])
       .agg(err=('didOvershoot', 'mean'), sw=('switchbackCount', 'mean')).reset_index())
w_err = sec.pivot_table(index='participant', columns='condition', values='err').dropna()
w_sw = sec.pivot_table(index='participant', columns='condition', values='sw').dropna()


def paired_secondary(wide):
    a = wide['Optimized'].to_numpy(float)
    b = wide['Android'].to_numpy(float)
    out = {'and': float(np.mean(b)), 'opt': float(np.mean(a)), 'diff': float(np.mean(a - b)),
           'p': float('nan')}
    try:
        out['p'] = float(wilcoxon(a, b).pvalue)
    except Exception:
        pass
    return out


ERR = paired_secondary(w_err)
SW = paired_secondary(w_sw)
print('Overshoot-Fehlerrate: And=%.3f  Opt=%.3f  diff=%+.3f  Wilcoxon p=%.4g' % (
    ERR['and'], ERR['opt'], ERR['diff'], ERR['p']))
print('Switchbacks pro Trial: And=%.3f  Opt=%.3f  diff=%+.3f  Wilcoxon p=%.4g' % (
    SW['and'], SW['opt'], SW['diff'], SW['p']))

piv = trials.pivot_table(index=['participant', 'distance'], columns='condition',
                         values='didOvershoot').dropna()
both = int(((piv['Optimized'] == 1) & (piv['Android'] == 1)).sum())
opt_only = int(((piv['Optimized'] == 1) & (piv['Android'] == 0)).sum())
and_only = int(((piv['Optimized'] == 0) & (piv['Android'] == 1)).sum())
neither = int(((piv['Optimized'] == 0) & (piv['Android'] == 0)).sum())
MCNEMAR_P = float('nan')
try:
    from statsmodels.stats.contingency_tables import mcnemar
    MCNEMAR_P = float(mcnemar([[both, opt_only], [and_only, neither]],
                              exact=(opt_only + and_only) < 25).pvalue)
except Exception as exc:
    print('McNemar nicht berechenbar:', exc)
print('Overshoot-Kreuztabelle: beide=%d  nur Opt=%d  nur And=%d  keiner=%d  McNemar p=%.4g' % (
    both, opt_only, and_only, neither, MCNEMAR_P))


Overshoot-Fehlerrate: And=0.570  Opt=0.780  diff=+0.210  Wilcoxon p=0.01562
Switchbacks pro Trial: And=0.250  Opt=0.470  diff=+0.220  Wilcoxon p=0.009766
Overshoot-Kreuztabelle: beide=51  nur Opt=27  nur And=6  keiner=16  McNemar p=0.0004985


## 8b. Overshoot in Abhaengigkeit von den Parametern

Waehrend der Studie testet jeder Proband in den Bloecken 1-18 viele verschiedene
Parameterkombinationen (je 10 Trials, jeweils alle Distanzen 30-300 genau einmal),
danach folgen Android- und optimierter Block. Daraus ergeben sich je Proband 18
**Bloecke mit erkundeten Parametern**, deren Overshoot direkt mit dem eigenen
Android-Block verglichen werden kann.

**Masse je Block:** Overshoot-Rate (Anteil Trials mit Overshoot) und Overshoot-Weite
(`maxOvershootDistancePx`, Mittel ueber alle 10 Trials; 0 bei Trials ohne Overshoot).

**A. Zusammenhang:** Spearman-Korrelation je Parameter (gepoolt ueber alle Bloecke und
*innerhalb* der Probanden, Wilcoxon auf den Einzelkorrelationen) sowie ein gemischtes
Modell mit allen drei Parametern gleichzeitig (z-standardisiert, $\mu$ logarithmiert,
Zufallsintercept je Proband), damit die Effekte nicht vermischt werden.

**B. Android vs. andere Werte:** Jeder Parameter wird in `N_BINS` Klassen eingeteilt
(Standard: Quantile der erkundeten Bloecke, damit jede Klasse gleich viele Bloecke hat
und die Skala keine Rolle spielt). Je Klasse wird der Overshoot pro Proband mit dessen
Android-Block gepaart verglichen (Wilcoxon, Holm-korrigiert ueber alle Tests).
Eigene Klassengrenzen lassen sich ueber `BIN_EDGES` vorgeben, z. B. relativ zum
Android-Wert (`{'scrollFriction': [0.0025, 0.015, 0.05, 0.15, 0.3]}`).
Die Auswertung ist bei wenigen Probanden explorativ und waechst mit jedem weiteren Teilnehmer.

In [ ]:
import warnings

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr

N_BINS = 3
BIN_EDGES = {}  # optional: {'scrollFriction': [...], 'decelerationRate': [...], 'inflexion': [...]}
MEASURES = {'rate': 'Overshoot-Rate', 'px': 'Overshoot-Weite (px)'}

# --- Block-Ebene: alle Bloecke aller Probanden ---
block_rows = []
for path in files:
    with open(path, encoding='utf-8') as fh:
        data = json.load(fh)
    for p in (data if isinstance(data, list) else [data]):
        conds = final_condition_blocks(p)
        if conds is None:
            continue
        android_run = conds['Android'].get('runNumber')
        opt_run = conds['Optimized'].get('runNumber')
        for b in p.get('attemptBlocks') or []:
            ps = b.get('parameterSet') or {}
            vals = [_num_or_nan(ps.get(k)) for k in PARAMS]
            ats = [a for a in (b.get('attempts') or []) if isinstance(a.get('didOvershoot'), bool)]
            if not ats or np.isnan(vals).any():
                continue
            role = ('Android' if b.get('runNumber') == android_run
                    else 'Optimized' if b.get('runNumber') == opt_run else 'Explored')
            block_rows.append({
                'participant': display_name(p), 'run': b.get('runNumber'), 'role': role,
                **dict(zip(PARAMS, vals)),
                'rate': float(np.mean([a['didOvershoot'] for a in ats])),
                'px': float(np.nanmean([_num_or_nan(a.get('maxOvershootDistancePx')) for a in ats])),
            })

blocks = pd.DataFrame(block_rows)
explored = blocks[blocks['role'] == 'Explored'].copy()
android_blocks = blocks[blocks['role'] == 'Android'].set_index('participant')
ANDROID_VALUES = {k: float(android_blocks[k].mean()) for k in PARAMS}
print('Bloecke gesamt: %d | erkundet: %d | Probanden: %d' % (
    len(blocks), len(explored), explored['participant'].nunique()))
print('Overshoot im Mittel: erkundet Rate=%.3f Weite=%.0f px | Android Rate=%.3f Weite=%.0f px' % (
    explored['rate'].mean(), explored['px'].mean(),
    android_blocks['rate'].mean(), android_blocks['px'].mean()))


def spearman_safe(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return float('nan'), float('nan')
    r, pv = spearmanr(x, y)
    return float(r), float(pv)


def wilcoxon_safe(a, b=None):
    try:
        return float(wilcoxon(a, b).pvalue)
    except Exception:
        return float('nan')


# --- A. Zusammenhang Parameter <-> Overshoot ---
OV_CORR = {}
corr_rows = []
for name in PARAMS:
    for m, m_label in MEASURES.items():
        rho, p_pool = spearman_safe(explored[name], explored[m])
        per_subject = [spearman_safe(g[name], g[m])[0] for _, g in explored.groupby('participant')]
        per_subject = [r for r in per_subject if not math.isnan(r)]
        within_rho = float(np.mean(per_subject)) if per_subject else float('nan')
        within_p = wilcoxon_safe(per_subject) if len(per_subject) >= 6 else float('nan')
        OV_CORR[(name, m)] = {'rho': rho, 'p': p_pool, 'within_rho': within_rho, 'within_p': within_p}
        corr_rows.append({'parameter': PARAM_LABELS[name], 'measure': m_label,
                          'rho_pooled': rho, 'p_pooled': p_pool,
                          'rho_within_mean': within_rho, 'p_within': within_p,
                          'n_subjects': len(per_subject)})
corr_table = pd.DataFrame(corr_rows)
print('\nA. Spearman-Korrelation (erkundete Bloecke):')
print(corr_table.round(4).to_string(index=False))

# Gemischtes Modell mit allen Parametern gleichzeitig (partielle Effekte)
mm = explored.copy()
mm['zMu'] = (np.log(mm['scrollFriction']) - np.log(mm['scrollFriction']).mean()) / np.log(mm['scrollFriction']).std()
mm['zR'] = (mm['decelerationRate'] - mm['decelerationRate'].mean()) / mm['decelerationRate'].std()
mm['zBeta'] = (mm['inflexion'] - mm['inflexion'].mean()) / mm['inflexion'].std()
OV_LMM = {}
lmm_rows = []
for m, m_label in MEASURES.items():
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fit = smf.mixedlm(m + ' ~ zMu + zR + zBeta', mm, groups=mm['participant']).fit(reml=True)
        for term, name in (('zMu', 'scrollFriction'), ('zR', 'decelerationRate'), ('zBeta', 'inflexion')):
            OV_LMM[(name, m)] = {'coef': float(fit.params[term]), 'p': float(fit.pvalues[term])}
            lmm_rows.append({'measure': m_label, 'parameter': PARAM_LABELS[name],
                             'coef_per_SD': float(fit.params[term]), 'p': float(fit.pvalues[term])})
    except Exception as exc:
        print('Gemischtes Modell (%s) nicht berechenbar: %s' % (m, exc))
if lmm_rows:
    print('\nA2. Gemischtes Modell (Effekt pro 1 SD des Parameters, partiell):')
    print(pd.DataFrame(lmm_rows).round(4).to_string(index=False))

# --- B. Android vs. Klassen anderer Parameterwerte ---
bin_rows = []
bin_defs = {}
for name in PARAMS:
    edges = BIN_EDGES.get(name)
    if edges is None:
        edges = np.unique(np.quantile(explored[name], np.linspace(0, 1, N_BINS + 1)))
    classes = pd.cut(explored[name], edges, include_lowest=True)
    bin_defs[name] = classes
    for interval, grp in explored.groupby(classes, observed=True):
        per_subject = grp.groupby('participant')[list(MEASURES)].mean()
        common = per_subject.index.intersection(android_blocks.index)
        for m, m_label in MEASURES.items():
            a = per_subject.loc[common, m]
            ref = android_blocks.loc[common, m]
            bin_rows.append({
                'parameter': PARAM_LABELS[name], 'key': name, 'measure': m, 'class': '%.3g - %.3g' % (interval.left, interval.right),
                'blocks': len(grp), 'subjects': len(common),
                'class_mean': float(a.mean()), 'android_mean': float(ref.mean()),
                'diff': float((a - ref).mean()), 'p': wilcoxon_safe(a, ref) if len(common) >= 6 else float('nan'),
            })
bins_table = pd.DataFrame(bin_rows)
valid = bins_table['p'].notna()
bins_table['p_holm'] = np.nan
if valid.any():
    bins_table.loc[valid, 'p_holm'] = multipletests(bins_table.loc[valid, 'p'], method='holm')[1]
print('\nB. Klasse vs. Android-Block (gepaart je Proband; Android: mu=%.3f, r=%.3f, beta=%.2f):' % (
    ANDROID_VALUES['scrollFriction'], ANDROID_VALUES['decelerationRate'], ANDROID_VALUES['inflexion']))
for m, m_label in MEASURES.items():
    print('\n' + m_label)
    print(bins_table[bins_table['measure'] == m].drop(columns=['key', 'measure']).round(4).to_string(index=False))

# --- Plot: Overshoot ueber Parameterwerten, Android als Referenz ---
fig, axes = plt.subplots(len(MEASURES), len(PARAMS), figsize=(14, 7), sharey='row')
for i, (m, m_label) in enumerate(MEASURES.items()):
    for j, name in enumerate(PARAMS):
        ax = axes[i, j]
        ax.scatter(explored[name], explored[m], s=10, alpha=0.25, color='#999999')
        sub = bins_table[(bins_table['key'] == name) & (bins_table['measure'] == m)]
        centers = [explored.loc[bin_defs[name] == iv, name].median() for iv in bin_defs[name].cat.categories
                   if (bin_defs[name] == iv).any()]
        ax.plot(centers, sub['class_mean'].to_numpy(), 'o-', color='#3b7dd8', label='Klassenmittel')
        ax.scatter([ANDROID_VALUES[name]], [android_blocks[m].mean()], marker='*', s=220, color='#d83b3b',
                   zorder=5, label='Android')
        if name == 'scrollFriction':
            ax.set_xscale('log')
        if i == len(MEASURES) - 1:
            ax.set_xlabel(PARAM_LABELS[name])
        if j == 0:
            ax.set_ylabel(m_label)
        if i == 0 and j == 0:
            ax.legend()
fig.suptitle('Overshoot in Abhaengigkeit von den Parametern (graue Punkte: Einzelbloecke)')
save_fig(fig, 'eval_overshoot_vs_params.png')
plt.show()

## 9. Vergleich nach Geschlecht

Der gepaarte log-Vergleich wird je Geschlechtsgruppe wiederholt (nur bei
mindestens `MIN_GROUP` Probanden), und das Pro-Proband-Delta der log-Zeiten wird
mit dem Mann-Whitney-U-Test zwischen den Gruppen verglichen. Bei zu kleiner
Gruppe werden die Werte als `NA` ausgewiesen.


In [14]:
subj = (trials.groupby(['participant', 'gender', 'condition'])['logTime']
        .mean().reset_index())
wide = subj.pivot_table(index=['participant', 'gender'], columns='condition',
                        values='logTime').dropna().reset_index()
wide['dlog'] = wide['Optimized'] - wide['Android']

GENDER = {}
for g in sorted(wide['gender'].unique()):
    sub = wide[wide['gender'] == g]
    if len(sub) >= MIN_GROUP:
        tt_g = ttest_rel(sub['Optimized'], sub['Android'])
        GENDER[g] = {'n': len(sub), 'ratio': math.exp(float(sub['dlog'].mean())),
                     'p': float(tt_g.pvalue)}
        print('  %-7s (n=%d): Verhaeltnis Opt/And=%.4f  gepaarter t-Test p=%.4g' % (
            g, len(sub), GENDER[g]['ratio'], GENDER[g]['p']))
    else:
        GENDER[g] = {'n': len(sub), 'ratio': float('nan'), 'p': float('nan')}
        print('  %-7s (n=%d): zu klein (<%d) - uebersprungen' % (g, len(sub), MIN_GROUP))

GENDER_MW_P = float('nan')
big = [g for g in GENDER if GENDER[g]['n'] >= MIN_GROUP]
if len(big) >= 2:
    a = wide[wide['gender'] == big[0]]['dlog']
    b = wide[wide['gender'] == big[1]]['dlog']
    try:
        GENDER_MW_P = float(stats.mannwhitneyu(a, b, alternative='two-sided').pvalue)
        print('  Mann-Whitney U auf dem Delta (%s vs %s): p=%.4g' % (big[0], big[1], GENDER_MW_P))
    except Exception as exc:
        print('  Mann-Whitney nicht berechenbar:', exc)
else:
    print('  Kein Zwischengruppen-Vergleich (zu wenige ausreichend grosse Gruppen).')

# Plot nur wenn mindestens eine Gruppe ausreichend gross ist.
plot_groups = [g for g in sorted(GENDER) if GENDER[g]['n'] >= MIN_GROUP]
if plot_groups:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    x = np.arange(len(plot_groups))
    w = 0.38
    and_gm = [math.exp(float(wide[wide.gender == g]['Android'].mean())) for g in plot_groups]
    opt_gm = [math.exp(float(wide[wide.gender == g]['Optimized'].mean())) for g in plot_groups]
    ax.bar(x - w / 2, and_gm, w, label='Android', color='#888888')
    ax.bar(x + w / 2, opt_gm, w, label='Optimized', color='#3b7dd8')
    ax.set_xticks(x)
    ax.set_xticklabels(['%s (n=%d)' % (g, GENDER[g]['n']) for g in plot_groups])
    ax.set_ylabel('Geometrisches Mittel der Zeit (ms)')
    ax.set_title('Android vs. Optimized nach Geschlecht')
    ax.legend()
    save_fig(fig, 'eval_android_vs_optimized_by_gender.png')
    plt.show()
else:
    print('Kein Geschlechter-Plot erzeugt (keine Gruppe >= %d).' % MIN_GROUP)


  female  (n=4): Verhaeltnis Opt/And=0.8946  gepaarter t-Test p=0.2278
  male    (n=6): Verhaeltnis Opt/And=0.8591  gepaarter t-Test p=0.07413
  Mann-Whitney U auf dem Delta (female vs male): p=0.6095


## 10. Personalisierung: Streuung der optimierten Parameter

Streuung der optimierten Parameter ueber alle Teilnehmer, relativ zur Breite des
Suchraums. Enge Clusterung spricht fuer eine gepoolte Funktion, breite Streuung
fuer Personalisierung. Zusaetzlich der elementweise **gepoolte Median** als
deskriptiver Vergleichspunkt (kein Performance-Vergleich -- dazu fehlt eine
Messung unter der gepoolten Funktion).


In [15]:
pdf = opt_params.dropna(subset=PARAMS).copy()
N_OPT = len(pdf)
print('Teilnehmer mit optimierten Parametern:', N_OPT)

SPREAD = {}
for name in PARAMS:
    vals = opt_params[name].dropna().to_numpy(float)
    if len(vals) == 0:
        continue
    lo, hi = PARAM_BOUNDS[name]
    width = hi - lo
    med = float(np.median(vals))
    iqr = float(np.percentile(vals, 75) - np.percentile(vals, 25))
    rng = float(vals.max() - vals.min())
    SPREAD[name] = {'median': med, 'iqr': iqr, 'range': rng,
                    'iqr_pct': 100 * iqr / width, 'range_pct': 100 * rng / width}
    print('  %-22s median=%.4f  IQR=%.4f (%.1f%% der Spanne)  range=%.4f (%.1f%%)' % (
        PARAM_LABELS[name], med, iqr, 100 * iqr / width, rng, 100 * rng / width))

if N_OPT >= 1:
    fig, axes = plt.subplots(1, len(PARAMS), figsize=(11, 4))
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, PARAMS):
        vals = opt_params[name].dropna().to_numpy(float)
        lo, hi = PARAM_BOUNDS[name]
        if len(vals) > 0:
            ax.boxplot(vals, vert=True, widths=0.5)
            ax.scatter(np.ones_like(vals), vals, alpha=0.5, color='#3b7dd8', zorder=3)
        ax.axhline(ANDROID_DEFAULTS[name], color='red', linestyle='--', label='Android default')
        ax.set_ylim(lo, hi)
        ax.set_title(PARAM_LABELS[name], fontsize=9)
        ax.set_xticks([])
        ax.legend(fontsize=7)
    axes[0].set_ylabel('Optimierter Wert (ueber zulaessigem Bereich)')
    fig.suptitle('Streuung der optimierten Parameter ueber %d Teilnehmer' % N_OPT)
    save_fig(fig, 'eval_optimized_param_spread.png')
    plt.show()

POOLED_MED = {name: float(np.median(opt_params[name].dropna())) for name in PARAMS}
dists = []
for _, row in opt_params.dropna(subset=PARAMS).iterrows():
    acc = 0.0
    for name in PARAMS:
        lo, hi = PARAM_BOUNDS[name]
        acc += ((row[name] - POOLED_MED[name]) / (hi - lo)) ** 2
    dists.append(math.sqrt(acc))
dists = np.array(dists) if dists else np.array([float('nan')])
POOLED_DIST = (float(np.mean(dists)), float(np.max(dists)))
print('Gepoolter Median-Parametersatz:', {k: round(v, 4) for k, v in POOLED_MED.items()})
print('Normierter Abstand zum gepoolten Median: mean=%.4f  max=%.4f' % POOLED_DIST)


Teilnehmer mit optimierten Parametern: 10
  mu (scrollFriction)    median=0.1310  IQR=0.1545 (51.9% der Spanne)  range=0.2970 (99.8%)
  r (decelerationRate)   median=12.6048  IQR=5.8028 (41.4% der Spanne)  range=10.9693 (78.4%)
  beta (inflexion)       median=0.8150  IQR=0.2225 (23.4% der Spanne)  range=0.5300 (55.8%)
Gepoolter Median-Parametersatz: {'scrollFriction': 0.131, 'decelerationRate': 12.6048, 'inflexion': 0.815}
Normierter Abstand zum gepoolten Median: mean=0.4457  max=0.7459


## 11. Zusammenfassung


In [16]:
print('=== Zusammenfassung (log-transformierte Zeiten) ===')
print('Teilnehmer:', N_PARTICIPANTS, '| Geschlechter:', gender_counts, '| gepaart (t-Test):', n)
print('Gepaarter t-Test (log): t=%.3f df=%d p=%.5f dz=%.3f (Wilcoxon p=%.4g)' % (
    tt.statistic, n - 1, tt.pvalue, dz, WILCOXON_P))
print('Verhaeltnis Opt/And=%.4f (95%% CI [%.4f, %.4f], %+.1f %%)' % (
    ratio, ratio_lo, ratio_hi, (ratio - 1) * 100))
print('GM Android=%.1f ms | GM Optimized=%.1f ms' % (GM['Android'], GM['Optimized']))
if 'Function' in PG:
    print('RM-ANOVA Funktion: F=%.3f p_unc=%.4g p_GG=%.4g' % (
        PG['Function']['F'], PG['Function']['p_unc'], PG['Function']['p_GG_corr']))
if 'Interaction' in PG:
    print('RM-ANOVA Interaktion: F=%.3f p_unc=%.4g p_GG=%.4g' % (
        PG['Interaction']['F'], PG['Interaction']['p_unc'], PG['Interaction']['p_GG_corr']))
print('Sekundaer: Fehlerrate diff=%+.3f (p=%.4g) | Switchbacks diff=%+.3f (p=%.4g)' % (
    ERR['diff'], ERR['p'], SW['diff'], SW['p']))


=== Zusammenfassung (log-transformierte Zeiten) ===
Teilnehmer: 10 | Geschlechter: {'male': 6, 'female': 4} | gepaart (t-Test): 10
Gepaarter t-Test (log): t=-2.839 df=9 p=0.01944 dz=-0.898 (Wilcoxon p=0.03711)
Verhaeltnis Opt/And=0.8731 (95% CI [0.7837, 0.9728], -12.7 %)
GM Android=4242.8 ms | GM Optimized=3704.6 ms
RM-ANOVA Funktion: F=8.059 p_unc=0.01944 p_GG=0.01944
RM-ANOVA Interaktion: F=1.717 p_unc=0.09838 p_GG=0.1785
Sekundaer: Fehlerrate diff=+0.210 (p=0.01562) | Switchbacks diff=+0.220 (p=0.009766)


## 12. CSV-Export fuer die Thesis

Alle Kennzahlen werden nach `plots/eval_results.csv` geschrieben (Spalten
`key,value,label,unit`). Die Thesis liest jeden Wert ueber seinen `key`
(`datatool`). CSV und Plots liegen im selben Ordner und sind austauschbar:
neue Daten -> Notebook erneut ausfuehren -> Thesis uebernimmt die Werte.
Keys sind camelCase (ohne Unterstriche), damit sie in LaTeX direkt nutzbar sind.


In [17]:
def _num(x, nd=2):
    try:
        xf = float(x)
    except (TypeError, ValueError):
        return 'NA'
    if math.isnan(xf):
        return 'NA'
    return f'{xf:.{nd}f}'


_rows = []


def add(key, value, label, unit=''):
    _rows.append({'key': key, 'value': value,
                  'label': label.replace(',', ';'), 'unit': unit.replace('%', 'pct')})


# Meta
add('nParticipants', str(int(N_PARTICIPANTS)), 'Number of evaluated participants')
add('nFemale', str(int(gender_counts.get('female', 0))), 'Number of female participants')
add('nMale', str(int(gender_counts.get('male', 0))), 'Number of male participants')
add('nFiles', str(len(files)), 'Number of result files')
add('nPaired', str(int(n)), 'Number of paired participants (t-test)')
add('nDistances', str(int(n_dist)), 'Number of target distances')

# Normalitaet / Transformation
add('skewRaw', _num(SKEW_RAW, 3), 'Skewness of raw timeMs')
add('skewLog', _num(SKEW_LOG, 3), 'Skewness of log timeMs')
add('shapiroRawP', _num(W_RAW[1], 4), 'Shapiro p pooled raw times')
add('shapiroLogP', _num(W_LOG[1], 4), 'Shapiro p pooled log times')
add('shapiroDiffW', _num(W_DIFF[0], 3), 'Shapiro W of paired log differences')
add('shapiroDiffP', _num(W_DIFF[1], 4), 'Shapiro p of paired log differences')
add('shapiroResidP', _num(RESID_SHAPIRO[1], 4), 'Shapiro p of ANOVA model residuals')

# Primaer: geometrische Mittel + gepaarter log-t-Test
add('gmAndroid', _num(GM['Android'], 1), 'Geometric mean time Android', 'ms')
add('gmAndroidCiLow', _num(GM_CI['Android'][0], 1), 'GM Android 95pct CI lower', 'ms')
add('gmAndroidCiHigh', _num(GM_CI['Android'][1], 1), 'GM Android 95pct CI upper', 'ms')
add('gmOptimized', _num(GM['Optimized'], 1), 'Geometric mean time Optimized', 'ms')
add('gmOptimizedCiLow', _num(GM_CI['Optimized'][0], 1), 'GM Optimized 95pct CI lower', 'ms')
add('gmOptimizedCiHigh', _num(GM_CI['Optimized'][1], 1), 'GM Optimized 95pct CI upper', 'ms')
add('meanLogDiff', _num(mean_d, 4), 'Mean paired log difference (Opt minus And)')
add('seLogDiff', _num(se, 4), 'Standard error of mean log difference')
add('logDiffCiLow', _num(ci_lo, 4), 'Log difference 95pct CI lower')
add('logDiffCiHigh', _num(ci_hi, 4), 'Log difference 95pct CI upper')
add('ratioOptAnd', _num(ratio, 4), 'Ratio of geometric means Opt over And')
add('ratioCiLow', _num(ratio_lo, 4), 'Ratio 95pct CI lower')
add('ratioCiHigh', _num(ratio_hi, 4), 'Ratio 95pct CI upper')
add('pctChange', _num((ratio - 1) * 100, 1), 'Percent change of time Opt vs And', '%')
add('pctChangeCiLow', _num((ratio_lo - 1) * 100, 1), 'Percent change CI lower', '%')
add('pctChangeCiHigh', _num((ratio_hi - 1) * 100, 1), 'Percent change CI upper', '%')
add('bootRatioCiLow', _num(BOOT_RATIO[0], 4), 'Bootstrap ratio CI lower')
add('bootRatioCiHigh', _num(BOOT_RATIO[1], 4), 'Bootstrap ratio CI upper')
add('tStat', _num(float(tt.statistic), 3), 'Paired t statistic (log time)')
add('tDf', str(int(n - 1)), 'Paired t degrees of freedom')
add('tP', _num(float(tt.pvalue), 4), 'Paired t-test p-value (log time)')
add('wilcoxonP', _num(WILCOXON_P, 4), 'Wilcoxon signed-rank p-value')
add('cohenDz', _num(dz, 3), 'Cohen dz effect size')

# RM-ANOVA (log) mit GG
for src_key, pfx in (('Function', 'anovaFunction'), ('Distance', 'anovaDistance'),
                     ('Interaction', 'anovaInteraction')):
    e = PG.get(src_key, {})
    add(pfx + 'F', _num(e.get('F'), 3), 'RM-ANOVA F %s' % src_key)
    add(pfx + 'Df1', _num(e.get('ddof1'), 0), 'RM-ANOVA df1 %s' % src_key)
    add(pfx + 'Df2', _num(e.get('ddof2'), 0), 'RM-ANOVA df2 %s' % src_key)
    add(pfx + 'P', _num(e.get('p_unc'), 4), 'RM-ANOVA uncorrected p %s' % src_key)
    add(pfx + 'Pgg', _num(e.get('p_GG_corr'), 4), 'RM-ANOVA Greenhouse-Geisser p %s' % src_key)
    add(pfx + 'Eps', _num(e.get('eps'), 3), 'Greenhouse-Geisser epsilon %s' % src_key)
    add(pfx + 'Np2', _num(e.get('np2'), 3), 'Partial eta squared %s' % src_key)
add('mauchlyDistanceW', _num(MAUCHLY[0], 4), 'Mauchly W for distance')
add('mauchlyDistanceP', _num(MAUCHLY[1], 4), 'Mauchly p for distance')
add('lmmFunctionP', _num(LMM_FUNCTION[1], 4), 'Mixed-model main effect of function p')
add('lmmFunctionRatio', _num(math.exp(LMM_FUNCTION[0]) if not math.isnan(LMM_FUNCTION[0]) else float('nan'), 4),
    'Mixed-model function effect as ratio')

# Sekundaermasse
add('errAndroid', _num(ERR['and'], 3), 'Overshoot error rate Android')
add('errOptimized', _num(ERR['opt'], 3), 'Overshoot error rate Optimized')
add('errDiff', _num(ERR['diff'], 3), 'Error-rate difference Opt minus And')
add('errWilcoxonP', _num(ERR['p'], 4), 'Error-rate Wilcoxon p-value')
add('swAndroid', _num(SW['and'], 3), 'Switchbacks per trial Android')
add('swOptimized', _num(SW['opt'], 3), 'Switchbacks per trial Optimized')
add('swDiff', _num(SW['diff'], 3), 'Switchback difference Opt minus And')
add('swWilcoxonP', _num(SW['p'], 4), 'Switchback Wilcoxon p-value')
add('mcnemarP', _num(MCNEMAR_P, 4), 'McNemar p-value paired overshoot')
add('overshootBoth', str(int(both)), 'Overshoot in both conditions count')
add('overshootOptOnly', str(int(opt_only)), 'Overshoot only Optimized count')
add('overshootAndOnly', str(int(and_only)), 'Overshoot only Android count')
add('overshootNeither', str(int(neither)), 'Overshoot in neither condition count')

# Geschlecht
for g, keyg in (('female', 'Female'), ('male', 'Male')):
    e = GENDER.get(g, {'n': 0, 'ratio': float('nan'), 'p': float('nan')})
    add('gender%sN' % keyg, str(int(e['n'])), 'Number of %s participants' % g)
    add('gender%sRatio' % keyg, _num(e['ratio'], 4), 'Ratio Opt over And %s' % g)
    add('gender%sP' % keyg, _num(e['p'], 4), 'Paired t-test p %s' % g)
add('genderDeltaMwP', _num(GENDER_MW_P, 4), 'Mann-Whitney U p on gender delta')

# Personalisierung
for name in PARAMS:
    kk = PARAM_KEY[name]
    e = SPREAD.get(name)
    if e is not None:
        add('spread%sMedian' % kk, _num(e['median'], 4), 'Median optimized %s' % PARAM_LABELS[name])
        add('spread%sIqrPct' % kk, _num(e['iqr_pct'], 1), 'IQR pct of range %s' % PARAM_LABELS[name], '%')
        add('spread%sRangePct' % kk, _num(e['range_pct'], 1), 'Range pct of range %s' % PARAM_LABELS[name], '%')
    add('pooledMedian%s' % kk, _num(POOLED_MED.get(name), 4), 'Pooled-median %s' % PARAM_LABELS[name])
add('pooledDistMean', _num(POOLED_DIST[0], 4), 'Mean normalized distance to pooled median')
add('pooledDistMax', _num(POOLED_DIST[1], 4), 'Max normalized distance to pooled median')

# Overshoot in Abhaengigkeit von den Parametern (erkundete Bloecke)
for name in PARAMS:
    kk = PARAM_KEY[name]
    for m, mk in (('rate', 'Rate'), ('px', 'Px')):
        e = OV_CORR.get((name, m), {})
        lab = '%s vs overshoot %s' % (PARAM_LABELS[name], 'rate' if m == 'rate' else 'distance')
        add('ov%sRho%s' % (mk, kk), _num(e.get('rho'), 3), 'Spearman rho pooled ' + lab)
        add('ov%sRhoP%s' % (mk, kk), _num(e.get('p'), 4), 'Spearman p pooled ' + lab)
        add('ov%sWithinRho%s' % (mk, kk), _num(e.get('within_rho'), 3), 'Mean within-participant Spearman rho ' + lab)
        add('ov%sWithinP%s' % (mk, kk), _num(e.get('within_p'), 4), 'Wilcoxon p of within-participant rho ' + lab)
        l = OV_LMM.get((name, m), {})
        add('ov%sLmmCoef%s' % (mk, kk), _num(l.get('coef'), 4), 'Mixed-model effect per SD ' + lab)
        add('ov%sLmmP%s' % (mk, kk), _num(l.get('p'), 4), 'Mixed-model p ' + lab)

# Overshoot: Referenzwerte und Klassen gegen den Android-Block
add('ovNBlocks', str(len(explored)), 'Number of explored blocks used for overshoot analysis')
add('ovNSubjects', str(int(explored['participant'].nunique())), 'Number of participants in overshoot analysis')
add('ovExploredRate', _num(explored['rate'].mean(), 3), 'Mean overshoot rate explored blocks')
add('ovExploredPx', _num(explored['px'].mean(), 1), 'Mean overshoot distance explored blocks', 'px')
add('ovAndroidRate', _num(android_blocks['rate'].mean(), 3), 'Mean overshoot rate Android block')
add('ovAndroidPx', _num(android_blocks['px'].mean(), 1), 'Mean overshoot distance Android block', 'px')
for name in PARAMS:
    add('ovAndroidValue%s' % PARAM_KEY[name], _num(ANDROID_VALUES[name], 4), 'Android default value %s' % PARAM_LABELS[name])
for name in PARAMS:
    kk = PARAM_KEY[name]
    for m, mk in (('rate', 'Rate'), ('px', 'Px')):
        sub = bins_table[(bins_table['key'] == name) & (bins_table['measure'] == m)].reset_index(drop=True)
        for c, row in sub.iterrows():
            pre = 'ovBin%s%sC%d' % (mk, kk, c + 1)
            lab = '%s class %d vs Android overshoot %s' % (PARAM_LABELS[name], c + 1, 'rate' if m == 'rate' else 'distance')
            add(pre + 'Range', row['class'].replace(' - ', '--'), 'Class range ' + lab)
            add(pre + 'Blocks', str(int(row['blocks'])), 'Blocks in ' + lab)
            add(pre + 'Mean', _num(row['class_mean'], 3 if m == 'rate' else 1), 'Class mean ' + lab)
            add(pre + 'Diff', _num(row['diff'], 3 if m == 'rate' else 1), 'Paired difference class minus Android ' + lab)
            add(pre + 'P', _num(row['p'], 4), 'Wilcoxon p ' + lab)
            add(pre + 'Pholm', _num(row['p_holm'], 4), 'Holm-adjusted p ' + lab)

results_df = pd.DataFrame(_rows, columns=['key', 'value', 'label', 'unit'])
csv_path = PLOTS_DIR / 'eval_results.csv'
try:
    results_df.to_csv(csv_path, index=False, encoding='utf-8')
    print('CSV geschrieben:', csv_path, '(', len(results_df), 'Werte )')
except OSError as exc:
    print('WARN: konnte CSV nicht schreiben:', exc)
results_df


CSV geschrieben: C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\plots\eval_results.csv ( 95 Werte )


,key,value,label,unit
0,nParticipants,10,Number of evaluated participants,
1,nFemale,4,Number of female participants,
2,nMale,6,Number of male participants,
3,nFiles,10,Number of result files,
4,nPaired,10,Number of paired participants (t-test),
...,...,...,...,...
90,spreadBetaIqrPct,23.4,IQR pct of range beta (inflexion),pct
91,spreadBetaRangePct,55.8,Range pct of range beta (inflexion),pct
92,pooledMedianBeta,0.8150,Pooled-median beta (inflexion),
93,pooledDistMean,0.4457,Mean normalized distance to pooled median,
